# TF-IDF + Logistic Regression Sentiment Classification

In [1]:
import pandas as pd
import numpy as np
import re
import warnings

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

warnings.filterwarnings('ignore')

## Data Loading

In [2]:
df = pd.read_csv("../../data/preprocessed/reviews_sentiment_analysis.csv")
print(f"Dataset size: {len(df)}")
print(f"\nSentiment distribution:")
print(df["positive=1/negative=0"].value_counts())
print(df["positive=1/negative=0"].value_counts(normalize=True))
df.head()

Dataset size: 260

Sentiment distribution:
positive=1/negative=0
1    135
0    125
Name: count, dtype: int64
positive=1/negative=0
1    0.519231
0    0.480769
Name: proportion, dtype: float64


,Review,positive=1/negative=0
0,Great food and great atmosphere! The chicken t...,1
1,I had heard good things about Tikka Shak so I ...,0
2,I was driving by tikka shack one day and decid...,0
3,Tikka Shack had the most modern and up-to-date...,1
4,Today is the third time I've come to India Pal...,1


## Preprocessing
TF-IDF works on a bag-of-words representation, so text preprocessing is applied to reduce noise.
Lowercasing, punctuation removal, stopword removal, and lemmatization are performed using NLTK.

In [3]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """
    Lowercases, removes punctuation, removes stopwords, and lemmatizes the text.
    Returns a single string of cleaned tokens.
    """
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 1]
    return " ".join(tokens)

df["cleaned_review"] = df["Review"].apply(preprocess_text)
df[["Review", "cleaned_review"]].head()

,Review,cleaned_review
0,Great food and great atmosphere! The chicken t...,great food great atmosphere chicken tikka masa...
1,I had heard good things about Tikka Shak so I ...,heard good thing tikka shak decided go ahead g...
2,I was driving by tikka shack one day and decid...,driving tikka shack one day decided give try l...
3,Tikka Shack had the most modern and up-to-date...,tikka shack modern uptodate atmosphere restaur...
4,Today is the third time I've come to India Pal...,today third time ive come india palace indian ...


## Data Split
Split the dataset into 80% training and 20% testing.
The data is shuffled and stratified by sentiment to maintain approximately equal positive and negative reviews in each split.

In [4]:
X = df["cleaned_review"]
y = df["positive=1/negative=0"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=101, shuffle=True, stratify=y
)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")
print(f"\nTrain class distribution:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))
print(f"\nTest class distribution:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))

Train size: 208
Test size: 52

Train class distribution:
positive=1/negative=0
1    108
0    100
Name: count, dtype: int64
positive=1/negative=0
1    0.519231
0    0.480769
Name: proportion, dtype: float64

Test class distribution:
positive=1/negative=0
1    27
0    25
Name: count, dtype: int64
positive=1/negative=0
1    0.519231
0    0.480769
Name: proportion, dtype: float64


## TF-IDF Vectorization
Fit the TF-IDF vectorizer on the training set and transform both training and test sets.

In [5]:
tfidf = TfidfVectorizer(max_features=1000, ngram_range=(1, 1), min_df=2, max_df=0.95)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"TF-IDF training matrix shape: {X_train_tfidf.shape}")
print(f"TF-IDF test matrix shape: {X_test_tfidf.shape}")

TF-IDF training matrix shape: (208, 750)
TF-IDF test matrix shape: (52, 750)


## Chi-Squared Feature Selection
Select the top features most associated with sentiment using the chi-squared statistical test, removing noisy or irrelevant features.

In [6]:
selector = SelectKBest(chi2, k=200)
X_train_selected = selector.fit_transform(X_train_tfidf, y_train)
X_test_selected = selector.transform(X_test_tfidf)

print(f"Features before chi-squared selection: {X_train_tfidf.shape[1]}")
print(f"Features after chi-squared selection: {X_train_selected.shape[1]}")

Features before chi-squared selection: 750
Features after chi-squared selection: 200


## Model Training
Train a Logistic Regression model with GridSearchCV to find the best hyperparameters.
5-fold cross-validation is used with F1-score as the scoring metric.

In [7]:
param_grid = {
    'C': [0.001, 0.01, 0.1, 1.0],
    'solver': ['liblinear'],
    'penalty': ['l1', 'l2'],
    'max_iter': [1000]
}

lr = GridSearchCV(
    LogisticRegression(random_state=101),
    param_grid,
    cv=5,
    scoring='f1'
)

lr.fit(X_train_selected, y_train)

print(f"Best Parameters: {lr.best_params_}")
print(f"Best CV F1-Score: {lr.best_score_:.4f}")

best_lr = lr.best_estimator_

Best Parameters: {'C': 1.0, 'max_iter': 1000, 'penalty': 'l2', 'solver': 'liblinear'}
Best CV F1-Score: 0.9379


## Evaluation
Evaluate the model on both the training and test sets.
Report accuracy and F1-score (macro average) for both.

In [8]:
train_pred = best_lr.predict(X_train_selected)
test_pred = best_lr.predict(X_test_selected)

print("Training Classification Report:")
print(classification_report(y_train, train_pred))

train_report = classification_report(y_train, train_pred, output_dict=True)
print(f"Training Accuracy: {train_report['accuracy']:.4f}")
print(f"Training F1-Score (macro): {train_report['macro avg']['f1-score']:.4f}\n")

print("Test Classification Report:")
print(classification_report(y_test, test_pred))

test_report = classification_report(y_test, test_pred, output_dict=True)
print(f"Test Accuracy: {test_report['accuracy']:.4f}")
print(f"Test F1-Score (macro): {test_report['macro avg']['f1-score']:.4f}")

Training Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       100
           1       0.98      0.98      0.98       108

    accuracy                           0.98       208
   macro avg       0.98      0.98      0.98       208
weighted avg       0.98      0.98      0.98       208

Training Accuracy: 0.9808
Training F1-Score (macro): 0.9807

Test Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.76      0.78        25
           1       0.79      0.81      0.80        27

    accuracy                           0.79        52
   macro avg       0.79      0.79      0.79        52
weighted avg       0.79      0.79      0.79        52

Test Accuracy: 0.7885
Test F1-Score (macro): 0.7878


## Results Summary

In [9]:
results = pd.DataFrame({
    "Set": ["Train", "Test"],
    "Accuracy": [
        round(train_report['accuracy'], 4),
        round(test_report['accuracy'], 4)
    ],
    "F1-Score (macro)": [
        round(train_report['macro avg']['f1-score'], 4),
        round(test_report['macro avg']['f1-score'], 4)
    ]
})

print("TF-IDF + Logistic Regression Results:")
results

TF-IDF + Logistic Regression Results:


,Set,Accuracy,F1-Score (macro)
0,Train,0.9808,0.9807
1,Test,0.7885,0.7878
